# DATA 2x01 Group Assignment 2026

**Topic:** NSW regional statistics, Greater Sydney POI data, SA2 resource scoring, and report analysis.

**Due:** 18 May 2026, 11:59 PM

This notebook is structured to document the full workflow required for the group submission.

## Deliverables Checklist

- PDF report
- Jupyter Notebook describing the full workflow
- Tutor conversation in Week 12 or Week 13
- One group ZIP file submitted to Canvas

## Setup

In [ ]:
from pathlib import Path
import json
import math
import sqlite3
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd()
CSV_PATH = DATA_DIR / "Region summary_ New South Wales STE 1.csv"
DB_PATH = DATA_DIR / "data2001_assignment.sqlite"

CSV_PATH

: 

# Task 1: NSW Summary Statistics

## 1.1 Load the NSW Region Summary CSV

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
df_raw.head()

In [ ]:
print("Shape:", df_raw.shape)
display(df_raw.info())
display(df_raw.describe(include="all"))

## 1.2 Data Cleaning

In [ ]:
df_clean = df_raw.copy()

# Standardise column names and text fields.
df_clean.columns = df_clean.columns.str.strip()
text_columns = ["Measure Code", "Parent Description", "Description"]
for column in text_columns:
    df_clean[column] = df_clean[column].astype("string").str.strip()

year_columns = [column for column in df_clean.columns if column.isdigit()]
df_clean[year_columns] = df_clean[year_columns].apply(pd.to_numeric, errors="coerce")

# Remove exact duplicate rows if present.
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print("Cleaned shape:", df_clean.shape)
df_clean.head()

In [ ]:
missing_summary = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df_clean) * 100).round(2)
missing_summary

### Cleaning Notes

Record the cleaning decisions here:

- Which columns were used as identifiers?
- Which years contain usable values?
- Were missing values meaningful, unavailable, or errors?
- Were duplicate rows found?
- Did any values require type conversion?

## 1.3 Derived Statistics

Each group member should contribute 5 derived statistics. Use this section to clearly label each member's work.

In [ ]:
latest_year = max(year_columns, key=int)
usable_years = [column for column in year_columns if df_clean[column].notna().any()]
latest_usable_year = max(usable_years, key=int)

print("All year columns:", year_columns)
print("Latest year column:", latest_year)
print("Latest usable year:", latest_usable_year)

### Ronnie Luo: Derived Statistics

Planned statistics:

1. Statistic 1
2. Statistic 2
3. Statistic 3
4. Statistic 4
5. Statistic 5

In [ ]:
# Example starting points for Task 1 statistics.
population_rows = df_clean[df_clean["Description"].str.contains("population", case=False, na=False)]
largest_latest_values = (
    df_clean[["Parent Description", "Description", latest_usable_year]]
    .dropna(subset=[latest_usable_year])
    .sort_values(latest_usable_year, ascending=False)
    .head(10)
)

display(population_rows.head(10))
display(largest_latest_values)

### Member 2: Derived Statistics

Planned statistics:

1. Statistic 1
2. Statistic 2
3. Statistic 3
4. Statistic 4
5. Statistic 5

### Member 3: Derived Statistics


Planned statistics:

1. Statistic 1
2. Statistic 2
3. Statistic 3
4. Statistic 4
5. Statistic 5

### Member 4: Derived Statistics


Planned statistics:

1. Statistic 1
2. Statistic 2
3. Statistic 3
4. Statistic 4
5. Statistic 5

### Additional Members

Copy the Member 1 structure for each group member. Each member should add 5 derived statistics and a short explanation of why each statistic is useful.

# Task 2: Greater Sydney SA2/SA4 Points of Interest Dataset

## 2.1 Select SA4 Zones

Each group member should select one distinct Greater Sydney SA4 zone.

Record the selected zones here:

- Member 1: SA4 zone name
- Member 2: SA4 zone name
- Member 3: SA4 zone name
- Member 4: SA4 zone name

In [ ]:
SELECTED_SA4_ZONES = [
    # "Sydney - City and Inner South",
    # "Parramatta",
]

# Add or load SA2 boundary data here when available.
# Expected columns: sa4_name, sa2_name, min_lon, min_lat, max_lon, max_lat
sa2_boundaries = pd.DataFrame(columns=["sa4_name", "sa2_name", "min_lon", "min_lat", "max_lon", "max_lat"])
sa2_boundaries

## 2.2 NSW Points of Interest API Function

In [ ]:
def fetch_pois_in_bbox(min_lon, min_lat, max_lon, max_lat, limit=1000):
    """Return POI records inside a bounding box.

    Update API_URL and params after confirming the NSW POI API endpoint from the Week 8 tutorial.
    """
    API_URL = "TODO_ADD_NSW_POI_API_ENDPOINT"
    params = {
        "min_lon": min_lon,
        "min_lat": min_lat,
        "max_lon": max_lon,
        "max_lat": max_lat,
        "limit": limit,
    }

    if API_URL.startswith("TODO"):
        return pd.DataFrame()

    url = API_URL + "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url) as response:
        payload = json.loads(response.read().decode("utf-8"))

    records = payload.get("features", payload if isinstance(payload, list) else [])
    return pd.json_normalize(records)


## 2.3 Loop Through SA2 Regions

In [ ]:
poi_frames = []

for _, row in sa2_boundaries.iterrows():
    if SELECTED_SA4_ZONES and row["sa4_name"] not in SELECTED_SA4_ZONES:
        continue

    pois = fetch_pois_in_bbox(row["min_lon"], row["min_lat"], row["max_lon"], row["max_lat"])
    if pois.empty:
        continue

    pois["sa4_name"] = row["sa4_name"]
    pois["sa2_name"] = row["sa2_name"]
    poi_frames.append(pois)

pois_df = pd.concat(poi_frames, ignore_index=True) if poi_frames else pd.DataFrame()
print("POI rows collected:", len(pois_df))
pois_df.head()

## 2.4 Store POI Dataset in Local Database

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    if not pois_df.empty:
        pois_df.to_sql("points_of_interest", conn, if_exists="replace", index=False)
        print("Saved points_of_interest table to", DB_PATH)
    else:
        print("No POI data saved yet. Complete the API endpoint and SA2 boundary inputs first.")

# Task 3: SA2 Well-Resourced Score

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def z_score(series):
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(0, index=series.index)
    return (series - series.mean()) / std

if not pois_df.empty and "sa2_name" in pois_df.columns:
    score_df = pois_df.groupby("sa2_name").size().rename("poi_count").reset_index()
    score_df["z_poi"] = z_score(score_df["poi_count"])
    score_df["score"] = sigmoid(score_df["z_poi"])
else:
    score_df = pd.DataFrame(columns=["sa2_name", "poi_count", "z_poi", "score"])

score_df.head()

## Scoring Explanation Notes

Use this section to explain:

- Why POI count is a reasonable proxy for resource availability
- Why z-score normalisation is used
- Why sigmoid scaling is used
- Whether population below 100 was excluded
- Any extensions to the formula, such as population adjustment or POI category weighting

# Task 4: Report Analysis and Visualisation

## 4.1 Key Findings from Task 1

Write the main statistical findings here after completing the derived statistics.

Possible angles:

- Population change over time
- Age structure
- Gender differences
- Density or growth indicators
- Measures with unusual changes or missingness

## 4.2 Score Visualisation Plan

Add plots here once `score_df` is populated.

Recommended visuals:

- Histogram of SA2 scores
- Top and bottom ranked SA2 regions
- Map overlay or choropleth if boundary geometry is available
- POI category breakdown by SA4 or SA2

In [ ]:
if not score_df.empty:
    display(score_df.sort_values("score", ascending=False).head(10))
    display(score_df.sort_values("score", ascending=True).head(10))
    display(score_df["score"].describe())
else:
    print("Score table is empty. Complete Task 2 before generating score summaries.")

## 4.3 Limitations

Discuss the limitations of the analysis here.

Possible limitations:

- POI count does not measure service quality or capacity
- Larger SA2s may naturally contain more POIs
- Population size may need to be considered
- API completeness and category definitions may affect results
- Bounding boxes can include POIs outside the actual SA2 polygon unless geometry filtering is added

# Next Steps

1. Add group member names and selected SA4 zones.
2. Complete Task 1 derived statistics.
3. Add SA2 boundary data for selected SA4 zones.
4. Confirm the NSW Points of Interest API endpoint from the Week 8 tutorial.
5. Store POI data in the local SQLite database.
6. Generate score visualisations and write report findings.